In [ ]:
!pip install datasets sentencepiece tqdm transformers


In [ ]:

import os, gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import sentencepiece as spm
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from transformers import pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ----------------------------
# 1. Load dataset (CNN/DailyMail)
# ----------------------------
dataset = load_dataset("cnn_dailymail", "3.0.0")

train_texts = dataset["train"]["article"][:5000]
train_summaries = dataset["train"]["highlights"][:5000]

test_texts = dataset["validation"]["article"][:200]
test_summaries = dataset["validation"]["highlights"][:200]

# ----------------------------
# 2. Train BPE tokenizer (smaller vocab = faster training)
# ----------------------------
os.makedirs("bpe", exist_ok=True)
with open("bpe/train.txt", "w", encoding="utf-8") as f:
    for text, summ in zip(train_texts, train_summaries):
        f.write(text.replace("\n", " ") + "\n")
        f.write(summ.replace("\n", " ") + "\n")

spm.SentencePieceTrainer.Train(
    input="bpe/train.txt",
    model_prefix="bpe/cnn_dailymail",
    vocab_size=4000,   # smaller than 8000
    model_type="bpe",
    character_coverage=0.9995,
    unk_id=0, pad_id=1, bos_id=2, eos_id=3
)

sp = spm.SentencePieceProcessor()
sp.load("bpe/cnn_dailymail.model")

PAD, BOS, EOS = sp.pad_id(), sp.bos_id(), sp.eos_id()
VOCAB_SIZE = sp.get_piece_size()
print("Tokenizer vocab size:", VOCAB_SIZE)

# ----------------------------
# 3. Encode helper with truncation
# ----------------------------
MAX_SRC_LEN = 400
MAX_TGT_LEN = 100

def encode_sentence(sentence, add_bos=False, add_eos=False, max_len=None):
    ids = sp.encode(sentence, out_type=int)
    if max_len: ids = ids[:max_len]
    if add_bos: ids = [BOS] + ids
    if add_eos: ids = ids + [EOS]
    return ids

# ----------------------------
# 4. Dataset + DataLoader
# ----------------------------
class SummarizationDataset(Dataset):
    def __init__(self, docs, sums):
        self.docs, self.sums = docs, sums

    def __len__(self): return len(self.docs)

    def __getitem__(self, idx):
        src = encode_sentence(self.docs[idx], add_eos=True, max_len=MAX_SRC_LEN)
        tgt_in = encode_sentence(self.sums[idx], add_bos=True, max_len=MAX_TGT_LEN)
        tgt_out = encode_sentence(self.sums[idx], add_eos=True, max_len=MAX_TGT_LEN)
        return torch.tensor(src), torch.tensor(tgt_in), torch.tensor(tgt_out)

def pad_sequence(seqs, pad_id=1):
    max_len = max(s.size(0) for s in seqs)
    out = torch.full((len(seqs), max_len), pad_id, dtype=torch.long)
    for i, s in enumerate(seqs):
        out[i, :s.size(0)] = s
    return out

def collate_fn(batch):
    srcs, tgts_in, tgts_out = zip(*batch)
    return (pad_sequence(srcs, PAD),
            pad_sequence(tgts_in, PAD),
            pad_sequence(tgts_out, PAD))

train_loader = DataLoader(SummarizationDataset(train_texts, train_summaries),
                          batch_size=4, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(SummarizationDataset(test_texts, test_summaries),
                         batch_size=2, shuffle=False, collate_fn=collate_fn)

# ----------------------------
# 5. Seq2Seq with Attention
# ----------------------------
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, pad_id):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.rnn = nn.GRU(emb_dim, hidden_dim, batch_first=True, bidirectional=True)

    def forward(self, src):
        emb = self.emb(src)
        outputs, hidden = self.rnn(emb)
        H = outputs.size(2)//2
        outputs = outputs[:,:,:H] + outputs[:,:,H:]
        hidden = hidden[0:hidden.size(0):2] + hidden[1:hidden.size(0):2]
        return outputs, hidden

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim*2, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, dec_hidden, enc_outputs):
        B, L, H = enc_outputs.shape
        dec = dec_hidden.unsqueeze(1).repeat(1,L,1)
        energy = torch.tanh(self.attn(torch.cat([dec, enc_outputs], dim=2)))
        scores = self.v(energy).squeeze(2)
        return F.softmax(scores, dim=1)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, pad_id, attention):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = attention
        self.rnn = nn.GRU(emb_dim+hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim*2, vocab_size)

    def forward(self, inp, hidden, enc_outputs):
        emb = self.emb(inp).unsqueeze(1)
        attn = self.attention(hidden[-1], enc_outputs)
        context = torch.bmm(attn.unsqueeze(1), enc_outputs)
        rnn_in = torch.cat([emb, context], dim=2)
        out, hidden = self.rnn(rnn_in, hidden)
        logits = self.fc(torch.cat([out, context], dim=2)).squeeze(1)
        return logits, hidden, attn

class Seq2Seq(nn.Module):
    def __init__(self, enc, dec, pad_id, bos_id, eos_id):
        super().__init__()
        self.enc, self.dec = enc, dec
        self.pad_id, self.bos_id, self.eos_id = pad_id, bos_id, eos_id

    def forward(self, src, tgt_in, tgt_out, teacher_forcing_ratio=0.9):  # more teacher forcing
        B, Lt = tgt_out.shape
        enc_out, hidden = self.enc(src)
        logits_seq = []
        inp = tgt_in[:,0]
        for t in range(Lt):
            logits, hidden, _ = self.dec(inp, hidden, enc_out)
            logits_seq.append(logits.unsqueeze(1))
            teacher = (torch.rand(1).item() < teacher_forcing_ratio)
            if t+1 < Lt:
                inp = tgt_in[:,t+1] if teacher else logits.argmax(dim=1)
        return torch.cat(logits_seq, dim=1)

    @torch.no_grad()
    def summarize(self, src, max_len=50):
        enc_out, hidden = self.enc(src)
        inp = torch.full((src.size(0),), self.bos_id, dtype=torch.long, device=src.device)
        outputs=[]
        for _ in range(max_len):
            logits, hidden, _ = self.dec(inp, hidden, enc_out)
            next_id = logits.argmax(dim=1)
            outputs.append(next_id)
            inp = next_id
            if torch.all(next_id==self.eos_id): break
        return torch.stack(outputs, dim=1)

# ----------------------------
# 6. Training with resume support
# ----------------------------
def train_model(model, loader, epochs=5, lr=1e-3, ckpt="seq2seq_ckpt.pt"):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD)
    model.to(device)

    # Resume if checkpoint exists
    if os.path.exists(ckpt):
        print("Resuming from checkpoint...")
        model.load_state_dict(torch.load(ckpt))

    for ep in range(1, epochs+1):
        model.train()
        ep_loss=0
        loop = tqdm(loader, desc=f"Epoch {ep}/{epochs}")
        for src,tin,tout in loop:
            src,tin,tout = src.to(device), tin.to(device), tout.to(device)
            optimizer.zero_grad()
            logits = model(src,tin,tout)
            B,Lt,V = logits.shape
            loss = criterion(logits.reshape(B*Lt,V), tout.reshape(B*Lt))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item()
            loop.set_postfix(loss=loss.item())
        print(f"Epoch {ep}, avg loss={ep_loss/len(loader):.4f}")
        torch.save(model.state_dict(), ckpt)  # save after each epoch

# ----------------------------
# 7. Run Training
# ----------------------------
enc = Encoder(VOCAB_SIZE, emb_dim=64, hidden_dim=128, pad_id=PAD)
attn = Attention(128)
dec = Decoder(VOCAB_SIZE, emb_dim=64, hidden_dim=128, pad_id=PAD, attention=attn)
model = Seq2Seq(enc, dec, PAD, BOS, EOS)

train_model(model, train_loader, epochs=5)

# ----------------------------
# 8. Test Summarization (custom model)
# ----------------------------
@torch.no_grad()
def summarize_text(text):
    model.eval()
    ids = encode_sentence(text, add_eos=True, max_len=MAX_SRC_LEN)
    src = torch.tensor(ids).unsqueeze(0).to(device)
    pred = model.summarize(src, max_len=40).squeeze(0).tolist()
    if EOS in pred: pred = pred[:pred.index(EOS)]
    return sp.decode(pred)

example = test_texts[0]
print("ARTICLE:", example[:400], "...")
print("REF SUMMARY:", test_summaries[0])
print("PRED SUMMARY (Custom):", summarize_text(example))

# ----------------------------
# 9. Test Summarization (Pretrained BART)
# ----------------------------
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=0 if device=="cuda" else -1)
print("PRED SUMMARY (BART):", summarizer(example, max_length=60, min_length=10, do_sample=False)[0]['summary_text'])


Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


PRED SUMMARY (BART): Zully Broussard gave one of her kidneys to a stranger. Her generosity paired up with big data. It resulted in six patients receiving transplants. The chain of surgeries is to be wrapped up Friday.
